# VecDB Bulk Loading – BYO Vectors (Manual IDs)

## 1. Scenario Overview
This notebook targets datasets that already contain dense vectors and explicit IDs. It creates a dense-vector table with `auto_generate_id=false`, bulk loads the BYO CSV, and inspects the resulting rows.

Upload `bulk_loading/data/bulktable_byov_ids.csv` to Object Storage and save the signed URL in `BYO_MANUAL_CSV_URL`.


## Architecture at a Glance

```text
BYO Vectors with Manual IDs

CSV payload contract
+----------------+----------------------+-------------------------------+
| ID             | DENSE_VECTOR          | METADATA                      |
+----------------+----------------------+-------------------------------+
| INC-2001       | [0.6394, ...]         | {ticket, service, sla, ...}   |
| INC-2002       | [0.4219, ...]         | {ticket, service, sla, ...}   |
+----------------+----------------------+-------------------------------+
        |                    |                         |
        |                    |                         |
        v                    v                         v
+------------------------------------------------------------------+
|                      VecDB bulk load                             |
|  source=BYO_MANUAL_CSV_URL  |  table=SUPPORT_BYO_IDS              |
|  ID policy: preserve IDs from CSV                                |
+-----------------------------+------------------------------------+
                              |
                              v
+------------------------------------------------------------------+
|                      Validation                                  |
|  list_vectors should return ticket IDs such as INC-2001,         |
|  with the same vector dimensions and metadata from the CSV.       |
+------------------------------------------------------------------+
```

This notebook validates the strict BYOV contract: the client owns IDs and vectors, while VecDB loads and serves them.


## What to Validate

- The CSV preview should show `ID`, `DENSE_VECTOR`, and `METADATA`.
- The vector dimension should match the table definition used for the BYO table.
- After loading, `list_vectors` should preserve the manual IDs from the CSV, such as `INC-2001`.


## 2. Setup





### Install Required Packages
Run the following cell once per environment to install / upgrade `oracle-vecdb`, `python-dotenv`, and `pandas`. Skip this step if you already have the dependencies available in your kernel.



In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas



### Load Credentials and Configure Demo Inputs
This cell reads `.env` for `VECDB_REST_URL`, `VECDB_USERNAME`/`VECDB_PASSWORD`, wires up the published Object Storage URLs, and defines table names plus the embedding model used later. Update the `.env` file or override any env var before running.



In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification (self-signed certificates).')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for this workflow.')

BYO_MANUAL_CSV_URL = os.getenv('BYO_MANUAL_CSV_URL')
if not BYO_MANUAL_CSV_URL:
    raise RuntimeError('Set BYO_MANUAL_CSV_URL with the Object Storage URL for this scenario.')
print('BYO manual CSV URL:', BYO_MANUAL_CSV_URL)

BYO_TABLE_MANUAL = os.getenv('BYO_TABLE_MANUAL')
print('BYO manual table:', BYO_TABLE_MANUAL)


### Local CSV Checkpoint

This checkpoint reads the CSV fixture from `bulk_loading/data/` only. It is not a VecDB response. Run the cell to preview the exact local file that should later be uploaded to Object Storage for the bulk load job.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


def _resolve_fixture(filename):
    for candidate in (Path('data') / filename, Path('bulk_loading') / 'data' / filename):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find {filename}. Run this notebook from bulk_loading/ or notebooks/vecdb/.')

fixture_path = _resolve_fixture('bulktable_byov_ids.csv')
fixture_df = pd.read_csv(fixture_path)
metadata_preview = fixture_df['METADATA'].apply(json.loads).apply(pd.Series)
vectors = fixture_df['DENSE_VECTOR'].apply(json.loads)
preview_df = pd.concat([fixture_df[['ID']], metadata_preview], axis=1)
preview_df['vector_dim'] = vectors.apply(len)
preview_df['vector_preview'] = vectors.apply(lambda values: values[:4])

scenario_summary = pd.DataFrame([
    ('Scenario', 'BYO vectors with manual IDs'),
    ('Local fixture', str(fixture_path)),
    ('Rows in fixture', len(fixture_df)),
    ('CSV columns', ', '.join(fixture_df.columns)),
    ('ID mode', 'IDs provided by CSV'),
    ('Vector source', 'DENSE_VECTOR column'),
    ('Vector dimension', len(vectors.iloc[0])),
    ('Target table', BYO_TABLE_MANUAL),
    ('Object Storage env var', 'BYO_MANUAL_CSV_URL'),
], columns=['Setting', 'Value'])

display(scenario_summary)
display(preview_df.head())


## 7. Bulk load vectors
Provision a BYO table that keeps the IDs supplied in the CSV. This mirrors the workflow where upstream systems already own vector generation and identifier management.



In [ ]:

print('Resetting BYO vectors table (manual IDs)...')
try:
    vecdb.drop_vector_table(name=BYO_TABLE_MANUAL)
    print(f'Dropped existing table {BYO_TABLE_MANUAL}')
except Exception:
    print(f'Table {BYO_TABLE_MANUAL} not present; creating new table')

vecdb.create_vector_table(
    name=BYO_TABLE_MANUAL,
    comment='Support tickets demo table with BYO vectors (manual IDs)',
    annotations={'DATASET': 'support_tickets'},
    table_params={"auto_generate_id":False},
)
print('Ready for BYO load:', BYO_TABLE_MANUAL)



## 8. Bulk Load BYO Dataset (Manual IDs)
Invoke `load_vectors` with the signed URL for `bulktable_byov_ids.csv`. The CSV already supplies dense vectors and IDs, so VecDB simply copies them into the BYO table.


In [ ]:
print('Submitting BYO manual bulk load job...')
byo_manual_job = vecdb.load_vectors(
    table_name=BYO_TABLE_MANUAL,
    url=BYO_MANUAL_CSV_URL,
)
BYO_MANUAL_JOB_NAME = getattr(byo_manual_job, 'job_name', None)
if BYO_MANUAL_JOB_NAME:
    print('BYO manual load job name:', BYO_MANUAL_JOB_NAME)
else:
    print('Raw BYO manual load job response:', byo_manual_job)



## 9. Inspect BYO Manual Load Job
Replay the job-inspection flow for the BYO ingest so reviewers can see consistent monitoring across scenarios.



In [ ]:
job_name = globals().get('BYO_MANUAL_JOB_NAME')
if job_name:
    details = vecdb.describe_vector_load_job(load_job_name=job_name)
    print('BYO manual job details:', details)
    try:
        logs = vecdb.get_vector_load_job_log(load_job_name=job_name)
        print('BYO manual log preview:', str(logs)[:500])
    except Exception as exc:
        print('Unable to fetch BYO manual load log:', exc)
else:
    print('Run the BYO manual load cell first to capture a job identifier.')



## 10. List BYO Manual Vectors
Query a handful of rows from `BYO_TABLE_MANUAL` to confirm that the IDs, vectors, and metadata match the CSV payload.


In [ ]:

byo_manual_vectors = vecdb.list_vectors(table_name=BYO_TABLE_MANUAL, limit=5)
print('Listed vectors count:', len(byo_manual_vectors.items))
for item in byo_manual_vectors.items:
    print(item.id, item.metadata, item.dense_vector)

## Cleanup
Drop the demo table so repeated runs stay isolated.

In [ ]:
print('Dropping BYO manual IDs table...')
try:
    vecdb.drop_vector_table(name=BYO_TABLE_MANUAL)
    print('Dropped', BYO_TABLE_MANUAL)
except Exception as exc:
    print('Unable to drop', BYO_TABLE_MANUAL, exc)